In [1]:
import nest_asyncio

nest_asyncio.apply()

print("Notebook Async Enabled!")

Notebook Async Enabled!


In [ ]:
import re
from typing import List
from pydantic import BaseModel
from langchain_openai import ChatOpenAI

def get_llm():

    return ChatOpenAI(
        model="Qwen/Qwen3-4B",
        base_url="http://0.0.0.0:8000/v1",
        api_key="abc-123",
        temperature=0
    )

llm = get_llm()

print("Connected to vLLM!")

In [ ]:
#Request Models & Domain Models

class AlarmRequest(BaseModel):
    device: str
    event: str
    severity: str

class AlarmAnalysis(BaseModel):
    incident_type: str
    severity: str
    event: str

class LogAnalysis(BaseModel):
    interface_down: bool
    bgp_neighbor_down: bool
    route_withdrawal: bool
    findings: List[str]

In [ ]:
#Agents

class AlarmAgent:

    async def run(
            self,
            alarm: AlarmRequest
    ) -> AlarmAnalysis:
        
        incident_type = "Unknown"

        if alarm.event == "BGP_NEIGHBOR_DOWN":

            incident_type = (
                "Network Routing Failure"
            )

        return AlarmAnalysis(
            incident_type=incident_type,
            severity=alarm.severity,
            event=alarm.event
        )
    
class LogAgent:
    """
    Parse router logs and extract findings
    """

    INTERFACE_PATTERN = re.compile(
        r"interface.*down",
        re.IGNORECASE
    )

    BGP_PATTERN = re.compile(
        r"bgp.*down",
        re.IGNORECASE
    )

    ROUTE_PATTERN = re.compile(
        r"withdraw",
        re.IGNORECASE
    )

    async def run(self, logs: list[str]) -> LogAnalysis:

        findings = []

        interface_down = False
        bgp_neighbor_down = False
        route_withdrawal = False

        for line in logs:

            if self.INTERFACE_PATTERN.search(line):
                interface_down = True
                findings.append("Interface Down")

            if self.BGP_PATTERN.search(line):
                bgp_neighbor_down = True
                findings.append("BGP Neighbor Down")

            if self.ROUTE_PATTERN.search(line):
                route_withdrawal = True
                findings.append("Route Withdrawal")

        findings = list(set(findings))

        return LogAnalysis(
            interface_down=interface_down,
            bgp_neighbor_down=bgp_neighbor_down,
            route_withdrawal=route_withdrawal,
            findings=findings
        )

In [5]:
alarm = AlarmRequest(
    device="RTR-01",
    event="BGP_NEIGHBOR_DOWN",
    severity="CRITICAL"
)

alarm

NameError: name 'AlarmRequest' is not defined

In [ ]:
logs = [

    "Jul 10 10:15:01 RTR01 Interface GigabitEthernet0/0 down",

    "Jul 10 10:15:03 RTR01 LINK-3-UPDOWN",

    "Jul 10 10:15:04 RTR01 BGP-5-ADJCHANGE neighbor 192.168.1.1 Down",

    "Jul 10 10:15:05 RTR01 Route withdrawal initiated"
]

logs

In [ ]:
alarm_agent = AlarmAgent()

alarm_result = await alarm_agent.run(
    alarm
)

alarm_result

In [ ]:
log_agent = LogAgent()

log_result = await log_agent.run(
    logs
)

log_result

In [ ]:
print("\n=== Alarm Analysis ===\n")

print(
    alarm_result.model_dump_json(
        indent=2
    )
)

print("\n=== Log Analysis ===\n")

print(
    log_result.model_dump_json(
        indent=2
    )
)